# OLS Linear Regression
OLS: ordinary least squares

## Example 1: does year of education increase income?
In the nls97 dataset, there is a variable called `highestgradecompleted.` It measures the highest grade of regular school the respondent has completed, typically ranging from 0 (no formal schooling) to 20 (completion of 8 or more years of higher education).

Regression equation:
$I_i = a + bE_i + \mu$, where $I_i$ represents income of individual $i$, and $E_i$ represents the years of formal school education.
Hypotheses:
- $H_0$: $b=0$
- $H_1$: $b \neq 0$

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

df = pd.read_csv("../05-data_cleaning/data/nls97.csv")
df.head()

### Prepare data and inspect data
- Cast the variables to numeric
- Drop missing values
- Remove outliers

In [ ]:
df['school_year'] = pd.to_numeric(df['highestgradecompleted'], errors='coerce')
df.dropna(subset=['school_year', 'wageincome'], inplace=True)
df['school_year'] = df['school_year'].astype(int)

In [ ]:
df[['school_year', 'wageincome']].describe()

In [ ]:

sns.set_style('whitegrid')
sns.scatterplot(data=df, x="school_year", y="wageincome")

We observe there some outliers with school year 95 and very high wageincome. They are the same values, so let's assume they are errors and remove them.

In [ ]:
df = df[(df['school_year']<40) & (df['wageincome']<200000)].copy()

In [ ]:
df[['school_year', 'wageincome']].describe()

In [ ]:
sns.set_style('whitegrid')
fig, axes=plt.subplots(nrows=1, ncols=3, figsize=(18, 5))

sns.scatterplot(data=df, x="school_year", y="wageincome", ax=axes[0], alpha=0.5)
sns.stripplot(data=df, x="school_year", y="wageincome", ax=axes[1], alpha=0.5, jitter=True)
sns.regplot(data=df, x='school_year', y='wageincome', ax=axes[2])

### Run regression


In [ ]:

X = df["school_year"]
y = df["wageincome"]

# Add constant (intercept)
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

### Interpretation
- $a=13580$, indicating people with no formal school year education still have income on average of \\$13,580.  This does not mean everyone who does not have education still earn \$13,580, it simply is the model intercept. (I usually try not to interpret the value of a).
- $b=2497.20$, indicating each extra year of school education increases income by \$2,497.
- $p < 0.001$ (in fact, p value can be access using `model.pvalues` and $p=6.85 \times 10^{-73}$, extremely small, so the coefficient $b$ is statistically significant.
- R-square: Proportion of variation in income explained by education

In [ ]:
alpha = 0.05

if model.pvalues["school_year"] < alpha:
    print("Reject H0: education significantly affects income")
else:
    print("Fail to reject H0")

In [ ]:
g0=sns.histplot(data=model.resid, bins=50)

In [ ]:
g1=sm.qqplot(model.resid, line='q')
plt.title("QQ Plot of Residuals")
plt.show()

## Example 2: multiple regression

Let's say, we also suspect income might be related to other variables.

${Income}_i = \beta_0 + \beta_1{Education}_i + \beta_2{Gender}_i+\beta{GPA}_i+\epsilon$


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm


df = pd.read_csv("../05-data_cleaning/data/nls97.csv")

numeric_vars = ['wageincome', 'highestgradecompleted', 'gpaoverall']

for var in numeric_vars:
    df[var] = pd.to_numeric(df[var], errors='coerce')

df.dropna(subset=numeric_vars + ['gender'], inplace=True)

df.head()

In [ ]:
df[numeric_vars].describe()

In [ ]:
df = df[(df['highestgradecompleted']<40) & (df['wageincome']<200000)].copy()
df = pd.get_dummies(data=df, columns=['gender'], drop_first=True, dtype=int)
df.head()

In [ ]:
# run regression

X = df[["highestgradecompleted", 'gpaoverall','gender_Male']].rename(columns={
    "highestgradecompleted":"school_year",
    "gpaoverall":"GPA",
    "gender_Male":"is_male"
})
y = df["wageincome"]

# Add constant (intercept)
X = sm.add_constant(X)

model2 = sm.OLS(y, X).fit()

print(model2.summary())

The large condition number could be caused by strong correlations between independent variables.

- Collinearity refers to a high linear correlation between two independent variables in a regression model.
- Multicollinearity refers to two or more independent variables highly correlated, or one variable is a linear combination of others

In [ ]:
from scipy.stats import pearsonr
from itertools import combinations

for var1, var2 in combinations(['highestgradecompleted',"gpaoverall","gender_Male"], 2):
    corr, p_value = pearsonr(df[var1], df[var2])
    print(f"{var1} + {var2}: {corr=:.3f}, {p_value=:.5f}")

We can also check VIFs. The Variance Inflation Factor (VIF) measures how much the variance of an estimated regression coefficient is increased due to collinearity with other predictors. A VIF of 1 indicates no correlation, while values above 5–10 indicate high multicollinearity that may require removing variables, as they inflate standard errors and reduce coefficient reliability

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                   for i in range(X.shape[1])]

print(vif_data)

Inspect residue

In [ ]:
sns.set_style('whitegrid')
g2=sns.histplot(data=model2.resid, bins=50)

In [ ]:
# Inspect qq plot (A Quantile-Quantile (Q-Q) plot is a graphical tool used to assess if a dataset follows a specific theoretical distribution (often normal) by plotting sample quantiles against theoretical quantiles.)

g3=sm.qqplot(model2.resid, line='q')
plt.title("QQ Plot of Residuals")
plt.show()

In [ ]:
fig, axes=plt.subplots(2, 2, figsize=(18, 10))

g0=sns.histplot(data=model.resid, bins=50, ax=axes.flat[0])
g0.text(x=20_000, y=300, s="Model 1: Residue distribution", fontsize=15)
g1=sm.qqplot(model.resid, line='q', ax=axes.flat[2])

g2=sns.histplot(data=model2.resid, bins=50,ax=axes.flat[1])
g2.text(x=5_500, y=230, s="Model 2: Residue distribution", fontsize=15)
g3=sm.qqplot(model2.resid, line='q', ax=axes.flat[3])